# Pruebas de Gold

Casos sintéticos en memoria y reconciliación de las tablas publicadas. No se crean ni se modifican tablas.

In [1]:
import json
import sys
from pathlib import Path
from pyspark.sql import functions as F

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'config' / 'gold.json').is_file()), None)
if project_root is None:
    raise FileNotFoundError('No se encuentra config/gold.json')
sys.path.insert(0, str(project_root / 'scripts'))
from gold_metrics import generar_gold
config = json.loads((project_root / 'config' / 'gold.json').read_text(encoding='utf-8'))
silver = json.loads((project_root / config['silver_config']).read_text(encoding='utf-8'))


## Casos controlados

A y B tienen medición en día 3. C no tiene medición en ese día. D tiene cero reproducciones. A también tiene una medición en día 1 que no debe entrar en el resumen.

In [2]:
from datetime import date
from math import isclose
p = date(2026, 11, 1)
d3 = date(2026, 11, 3)
pubs = spark.createDataFrame([
    ('tiktok', 'A', p, 'regalos', 'A'), ('tiktok', 'B', p, 'regalos', 'B'),
    ('tiktok', 'C', p, 'regalos', 'C'), ('tiktok', 'D', p, 'regalos', 'D'),
    ('instagram', 'A', p, 'belleza', 'otro A'),
], 'plataforma string, video_id string, fecha_publicacion date, categoria string, titulo string')
meds = spark.createDataFrame([
    ('tiktok', 'A', p, 2, 50, 5, 0, 0),
    ('tiktok', 'A', d3, 1, 100, 8, 1, 1),
    ('tiktok', 'B', d3, 1, 300, 10, 3, 2),
    ('tiktok', 'C', p, 1, 100, 5, 0, 0),
    ('tiktok', 'D', d3, 1, 0, 1, 0, 0),
], 'plataforma string, video_id string, fecha_medicion date, version_origen int, reproducciones long, me_gusta long, comentarios long, compartidos long')
result = generar_gold(pubs, meds, 3)
v = result['rendimiento_video']
assert v.count() == 5
assert v.filter(F.col('fecha_medicion') == F.lit(p)).filter(F.col('dia_video') != 1).count() == 0
assert v.filter(F.col('reproducciones') == 0).first().tasa_interaccion is None
r = result['rendimiento_categoria'].filter(F.col('plataforma') == 'tiktok').first()
assert (r.videos_publicados, r.videos_medidos, r.videos_con_tasa, r.videos_sin_medicion, r.videos_sin_tasa) == (4, 3, 2, 1, 1)
assert r.reproducciones == 400 and r.interacciones == 25
assert isclose(r.tasa_interaccion, 6.25) and isclose(r.cobertura_pct, 75.0)
empty = result['rendimiento_plataforma'].filter(F.col('plataforma') == 'instagram').first()
assert empty.videos_medidos == 0 and empty.tasa_interaccion is None
print('Casos sintéticos correctos: día 3, cobertura, ratio ponderado, cero vistas y grupo sin mediciones.')

Casos sintéticos correctos: día 3, cobertura, ratio ponderado, cero vistas y grupo sin mediciones.


## Integridad de claves

Las duplicidades y referencias desconocidas deben detener la transformación.

In [3]:
for label, bad_pubs, bad_meds in [
    ('publicación duplicada', pubs.unionByName(pubs.limit(1)), meds),
    ('medición duplicada', pubs, meds.unionByName(meds.limit(1))),
    ('referencia desconocida', pubs, meds.withColumn('video_id', F.lit('Z'))),
]:
    try:
        generar_gold(bad_pubs, bad_meds)
    except ValueError as exc:
        print(label, ':', exc)
    else:
        raise AssertionError(f'No se detectó: {label}')

publicación duplicada : Claves duplicadas en publicaciones: ['plataforma', 'video_id']
medición duplicada : Claves duplicadas en mediciones vigentes: ['plataforma', 'video_id', 'fecha_medicion']
referencia desconocida : Claves duplicadas en mediciones vigentes: ['plataforma', 'video_id', 'fecha_medicion']


## Reconciliación de tablas reales

Comparar las tablas persistidas con una reconstrucción de solo lectura y validar cobertura y granularidad. Ejecutar después del notebook Gold y sin cargas concurrentes.

In [ ]:
publicaciones = spark.table(silver['tables']['publicaciones'])
mediciones = spark.table(silver['tables']['mediciones_vigentes'])
expected = generar_gold(publicaciones, mediciones, config['dia_objetivo'])
keys = {'rendimiento_video': ['plataforma', 'video_id', 'fecha_medicion'],
        'rendimiento_categoria': ['plataforma', 'categoria', 'dia_video'],
        'rendimiento_plataforma': ['plataforma', 'dia_video']}
for nombre, df in expected.items():
    actual = spark.table(config['tables'][nombre]).select(*df.columns)
    assert actual.count() == actual.select(*keys[nombre]).distinct().count(), nombre
    assert actual.exceptAll(df).limit(1).count() == 0, nombre
    assert df.exceptAll(actual).limit(1).count() == 0, nombre
    print(nombre, ':', actual.count(), 'filas reconciliadas')
video = spark.table(config['tables']['rendimiento_video'])
assert video.count() == mediciones.count()
for nombre in ['rendimiento_categoria', 'rendimiento_plataforma']:
    rows = spark.table(config['tables'][nombre])
    assert rows.agg(F.sum('videos_publicados')).first()[0] == publicaciones.count()
    assert rows.agg(F.sum('videos_medidos')).first()[0] == video.filter(F.col('dia_video') == config['dia_objetivo']).count()
    assert rows.filter(F.col('videos_medidos') > F.col('videos_publicados')).count() == 0
    assert rows.filter(F.col('videos_con_tasa') > F.col('videos_medidos')).count() == 0
print('Reconciliación Gold correcta.')